# Работа 5. Polars, ускорение pandas и оптимизация типов

Три независимых раздела на двух датасетах: `train.csv` («Титаник») и `Housing.csv`.
Оба лежат в общей папке `DataSet/` рядом с папками работ, поэтому путь ищется
вверх по дереву каталогов.

In [1]:
from pathlib import Path


def find_dataset(filename: str) -> Path:
    """Ищет DataSet/<filename>, поднимаясь от текущей папки вверх по дереву."""
    start = Path.cwd()
    for folder in [start, *start.parents]:
        candidate = folder / "DataSet" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Не найдена папка DataSet с файлом {filename}")


TITANIC = find_dataset("train.csv")
HOUSING = find_dataset("Housing.csv")
print(TITANIC)
print(HOUSING)

/Users/a8ee/Desktop/course/DataSet/train.csv
/Users/a8ee/Desktop/course/DataSet/Housing.csv


# Раздел 1. Polars

## 1.1. Чтение датасета с помощью polars

In [2]:
import polars as pl

# В polars нет индекса как в pandas: PassengerId остаётся обычным столбцом
titanic = pl.read_csv(TITANIC)
print(f"строк: {titanic.height}, столбцов: {titanic.width}")
titanic.head()

строк: 891, столбцов: 12


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


## 1.2. Основная информация о датасете

In [3]:
# Типы данных: schema отдаёт пары «столбец -> тип»
print("--- Типы данных ---")
for name, dtype in titanic.schema.items():
    print(f"  {name:<12} {dtype}")

--- Типы данных ---
  PassengerId  Int64
  Survived     Int64
  Pclass       Int64
  Name         String
  Sex          String
  Age          Float64
  SibSp        Int64
  Parch        Int64
  Ticket       String
  Fare         Float64
  Cabin        String
  Embarked     String


In [4]:
# Пропуски: null_count() возвращает одну строку с числом пропусков по каждому столбцу.
# Разворачиваем её вертикально, чтобы читалось как таблица.
nulls = titanic.null_count().transpose(include_header=True,
                                       header_name="столбец",
                                       column_names=["пропусков"])
nulls = nulls.with_columns(
    (pl.col("пропусков") / titanic.height * 100).round(2).alias("процент")
)
nulls

столбец,пропусков,процент
str,u32,f64
"""PassengerId""",0,0.0
"""Survived""",0,0.0
"""Pclass""",0,0.0
"""Name""",0,0.0
"""Sex""",0,0.0
…,…,…
"""Parch""",0,0.0
"""Ticket""",0,0.0
"""Fare""",0,0.0


In [5]:
# Средние значения по числовым столбцам.
# pl.col(pl.Int64, pl.Float64) выбирает столбцы по типу, .mean() применяется к каждому.
titanic.select(pl.col(pl.Int64, pl.Float64).mean()).transpose(
    include_header=True, header_name="столбец", column_names=["среднее"]
)

столбец,среднее
str,f64
"""PassengerId""",446.0
"""Survived""",0.383838
"""Pclass""",2.308642
"""Age""",29.699118
"""SibSp""",0.523008
"""Parch""",0.381594
"""Fare""",32.204208


In [6]:
# describe() — сводная статистика: count, null_count, mean, std, квартили
titanic.describe()

statistic,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


## 1.3. Количество пассажиров каждого класса

In [7]:
# get_column() достаёт столбец как Series, value_counts() считает повторения
passengers_by_class = (
    titanic.get_column("Pclass")
    .value_counts()
    .sort("Pclass")
    .rename({"count": "пассажиров"})
)
passengers_by_class

Pclass,пассажиров
i64,u32
1,216
2,184
3,491


## 1.4. Количество выживших мужчин и женщин

In [8]:
# Survived хранит 0 и 1, поэтому сумма столбца — это число выживших.
# len() внутри agg() даёт общее число пассажиров группы.
survivors = (
    titanic.group_by("Sex")
    .agg(
        pl.col("Survived").sum().alias("выжило"),
        pl.len().alias("всего"),
    )
    .with_columns(
        (pl.col("выжило") / pl.col("всего") * 100).round(2).alias("процент")
    )
    .sort("выжило", descending=True)
)
survivors

Sex,выжило,всего,процент
str,i64,u32,f64
"""female""",233,314,74.2
"""male""",109,577,18.89


## 1.5. Пассажиры старше 44 лет

In [9]:
# filter() оставляет строки, где выражение истинно.
# Пропуски в Age не пройдут фильтр: сравнение с null даёт null, а не True.
older_44 = titanic.filter(pl.col("Age") > 44)
print(f"найдено пассажиров: {older_44.height}")
older_44.head(10)

найдено пассажиров: 115


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
7,0,1,"""McCarthy, Mr. Timothy J""","""male""",54.0,0,0,"""17463""",51.8625,"""E46""","""S"""
12,1,1,"""Bonnell, Miss. Elizabeth""","""female""",58.0,0,0,"""113783""",26.55,"""C103""","""S"""
16,1,2,"""Hewlett, Mrs. (Mary D Kingcome…","""female""",55.0,0,0,"""248706""",16.0,null,"""S"""
34,0,2,"""Wheadon, Mr. Edward H""","""male""",66.0,0,0,"""C.A. 24579""",10.5,null,"""S"""
53,1,1,"""Harper, Mrs. Henry Sleeper (My…","""female""",49.0,1,0,"""PC 17572""",76.7292,"""D33""","""C"""
55,0,1,"""Ostby, Mr. Engelhart Cornelius""","""male""",65.0,0,1,"""113509""",61.9792,"""B30""","""C"""
63,0,1,"""Harris, Mr. Henry Birkhardt""","""male""",45.0,1,0,"""36973""",83.475,"""C83""","""S"""
93,0,1,"""Chaffee, Mr. Herbert Fuller""","""male""",46.0,1,0,"""W.E.P. 5734""",61.175,"""E31""","""S"""
95,0,3,"""Coxon, Mr. Daniel""","""male""",59.0,0,0,"""364500""",7.25,null,"""S"""


# Раздел 2. Ускорение работы с pandas

## 2.1. Чтение датасета с помощью pandas

In [10]:
import numpy as np
import pandas as pd

df = pd.read_csv(TITANIC, index_col="PassengerId")
print(f"строк: {df.shape[0]}, столбцов: {df.shape[1]}")
df.head()

строк: 891, столбцов: 11


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2.2. Средний возраст и стандартное отклонение через bottleneck

In [11]:
import bottleneck as bn

age = df["Age"].to_numpy()          # bottleneck работает с массивами numpy, не с Series

mean_age = bn.nanmean(age)
std_age = bn.nanstd(age, ddof=1)    # ddof=1 — выборочное стандартное отклонение

print(f"средний возраст:            {mean_age:.4f}")
print(f"стандартное отклонение:     {std_age:.4f}")
print(f"учтено значений:            {np.count_nonzero(~np.isnan(age))} из {len(age)}")

средний возраст:            29.6991
стандартное отклонение:     14.5265
учтено значений:            714 из 891


In [12]:
# Сверяем с pandas и обращаем внимание на ddof: у bottleneck по умолчанию ddof=0
# (стандартное отклонение генеральной совокупности), у pandas — ddof=1 (выборочное).
print(f"bn.nanstd(ddof=0): {bn.nanstd(age):.4f}   <- значение по умолчанию, отличается")
print(f"bn.nanstd(ddof=1): {bn.nanstd(age, ddof=1):.4f}")
print(f"pandas .std():     {df['Age'].std():.4f}")
print(f"совпадает с pandas: {np.isclose(bn.nanstd(age, ddof=1), df['Age'].std())}")

bn.nanstd(ddof=0): 14.5163   <- значение по умолчанию, отличается
bn.nanstd(ddof=1): 14.5265
pandas .std():     14.5265
совпадает с pandas: True


In [13]:
import timeit


def benchmark(label: str, function, number: int) -> dict:
    """Минимальное время одного вызова из пяти серий замеров, в микросекундах."""
    seconds = min(timeit.repeat(function, number=number, repeat=5)) / number
    return {"способ": label, "мкс на вызов": round(seconds * 1e6, 2)}


results = []
for name, array, number in (
    ("исходный массив (891 значение)", age, 2000),
    ("раздутый массив (1.78 млн)", np.tile(age, 2000), 20),
):
    series = pd.Series(array)
    for label, function in (
        ("bottleneck  bn.nanmean", lambda a=array: bn.nanmean(a)),
        ("numpy       np.nanmean", lambda a=array: np.nanmean(a)),
        ("pandas      .mean()", lambda s=series: s.mean()),
        ("bottleneck  bn.nanstd", lambda a=array: bn.nanstd(a, ddof=1)),
        ("numpy       np.nanstd", lambda a=array: np.nanstd(a, ddof=1)),
        ("pandas      .std()", lambda s=series: s.std()),
    ):
        results.append({"массив": name, **benchmark(label, function, number)})

pd.DataFrame(results).pivot(index="способ", columns="массив",
                            values="мкс на вызов")

массив,исходный массив (891 значение),раздутый массив (1.78 млн)
способ,,
bottleneck bn.nanmean,1.47,2798.79
bottleneck bn.nanstd,3.18,6167.31
numpy np.nanmean,7.36,2928.73
numpy np.nanstd,17.16,6653.23
pandas .mean(),7.46,2337.96
pandas .std(),6.01,6179.26


**Где bottleneck действительно ускоряет.** На реальном размере датасета (891 значение) `bn.nanmean` работает примерно в 5 раз быстрее, чем `np.nanmean` и `pandas.mean()`. Выигрыш здесь не в самом вычислении, а в накладных расходах: numpy при обработке пропусков строит промежуточную маску и делает несколько проходов по массиву, тогда как bottleneck выполняет специализированный цикл на C за один проход.\n\nНа раздутом массиве в 1.78 млн значений преимущество исчезает: там время определяется уже не накладными расходами, а скоростью чтения данных из памяти, и все три библиотеки упираются в одно и то же ограничение.\n\nОтсюда практический вывод: bottleneck выгоден на частых вызовах по небольшим массивам — например, в цикле по группам или по окнам — а не на одном вычислении по огромному столбцу.

## 2.3. Новый столбец `Fare_new` = `Fare` × 1.3

In [14]:
# Способ из задания: обход строк через itertuples()
def fare_via_itertuples(frame: pd.DataFrame) -> list[float]:
    return [row.Fare * 1.3 for row in frame.itertuples()]


# Способ из задания: apply() по столбцу
def fare_via_apply(frame: pd.DataFrame) -> pd.Series:
    return frame["Fare"].apply(lambda fare: fare * 1.3)


df["Fare_new"] = fare_via_itertuples(df)

# Проверяем, что все три способа дают один результат
assert np.allclose(df["Fare_new"], fare_via_apply(df))
assert np.allclose(df["Fare_new"], df["Fare"] * 1.3)

df[["Fare", "Fare_new"]].head()

,Fare,Fare_new
PassengerId,,
1,7.2500,9.42500
2,71.2833,92.66829
3,7.9250,10.30250
4,53.1000,69.03000
5,8.0500,10.46500


In [15]:
# Сравнение скорости трёх способов на том же датафрейме
print("itertuples():")
%timeit -n 10 -r 3 fare_via_itertuples(df)
print("apply():")
%timeit -n 10 -r 3 fare_via_apply(df)
print("векторизация (df['Fare'] * 1.3):")
%timeit -n 10 -r 3 df["Fare"] * 1.3

itertuples():
1.34 ms ± 14.6 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)
apply():
110 μs ± 6.39 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)
векторизация (df['Fare'] * 1.3):
23.9 μs ± 3.98 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


# Раздел 3. Оптимизация типов pandas

## 3.1. Чтение датасета Housing.csv

In [16]:
housing = pd.read_csv(HOUSING)
print(f"строк: {housing.shape[0]}, столбцов: {housing.shape[1]}")
housing.head()

строк: 545, столбцов: 13


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## 3.2. Выбор оптимального типа для каждого столбца

In [17]:
# Смотрим, что вообще лежит в столбцах: границы значений и число уникальных
info = pd.DataFrame({
    "тип": housing.dtypes.astype(str),
    "уникальных": housing.nunique(),
    "минимум": [housing[c].min() for c in housing.columns],
    "максимум": [housing[c].max() for c in housing.columns],
    "память, КБ": (housing.memory_usage(deep=True)[1:] / 1024).round(1),
})
info

,тип,уникальных,минимум,максимум,"память, КБ"
price,int64,219,1750000,13300000,4.3
area,int64,284,1650,16200,4.3
bedrooms,int64,6,1,6,4.3
bathrooms,int64,4,1,4,4.3
stories,int64,4,1,4,4.3
mainroad,str,2,no,yes,27.6
guestroom,str,2,no,yes,27.2
basement,str,2,no,yes,27.3
hotwaterheating,str,2,no,yes,27.2
airconditioning,str,2,no,yes,27.3


In [18]:
# ВЫВОДЫ ПО КАЖДОМУ СТОЛБЦУ
#
# Числовые: pandas по умолчанию берёт int64 (8 байт на значение) независимо от
# реального диапазона. Подбираем минимальный целый тип, вмещающий максимум:
#
#   price       1 750 000 .. 13 300 000  -> int32   (вмещает до 2 147 483 647)
#   area            1 650 .. 16 200      -> int16   (вмещает до 32 767)
#   bedrooms            1 .. 6           -> int8    (вмещает до 127)
#   bathrooms           1 .. 4           -> int8
#   stories             1 .. 4           -> int8
#   parking             0 .. 3           -> int8
#
# Строковые: шесть столбцов хранят только "yes"/"no" — это булев признак,
# и bool занимает 1 байт вместо целой строки в памяти:
#
#   mainroad, guestroom, basement,
#   hotwaterheating, airconditioning, prefarea   -> bool
#
# furnishingstatus принимает три значения (furnished / semi-furnished /
# unfurnished). Булевым его не сделать, но category хранит коды по 1 байту
# плюс один словарь категорий на весь столбец:
#
#   furnishingstatus  -> category
#
# Итог: ни одному столбцу не нужен int64, а строки не нужны вообще.

INT_TYPES = {"price": "int32", "area": "int16", "bedrooms": "int8",
             "bathrooms": "int8", "stories": "int8", "parking": "int8"}
BOOL_COLUMNS = ["mainroad", "guestroom", "basement",
                "hotwaterheating", "airconditioning", "prefarea"]

for column, dtype in INT_TYPES.items():
    print(f"{column:<18} {str(housing[column].dtype):<8} -> {dtype}")
for column in BOOL_COLUMNS:
    print(f"{column:<18} {str(housing[column].dtype):<8} -> bool")
print(f"{'furnishingstatus':<18} {str(housing['furnishingstatus'].dtype):<8} -> category")

price              int64    -> int32
area               int64    -> int16
bedrooms           int64    -> int8
bathrooms          int64    -> int8
stories            int64    -> int8
parking            int64    -> int8
mainroad           str      -> bool
guestroom          str      -> bool
basement           str      -> bool
hotwaterheating    str      -> bool
airconditioning    str      -> bool
prefarea           str      -> bool
furnishingstatus   str      -> category


## 3.3. Смена типов и сравнение потребления памяти

In [19]:
before = housing.memory_usage(deep=True).sum()

optimized = housing.astype(INT_TYPES)
optimized[BOOL_COLUMNS] = optimized[BOOL_COLUMNS].eq("yes")     # "yes"/"no" -> True/False
optimized["furnishingstatus"] = optimized["furnishingstatus"].astype("category")

after = optimized.memory_usage(deep=True).sum()

# Значения не должны измениться — проверяем несколько столбцов
assert (optimized["price"] == housing["price"]).all()
assert (optimized["mainroad"] == (housing["mainroad"] == "yes")).all()
assert (optimized["furnishingstatus"].astype(str) == housing["furnishingstatus"]).all()

comparison = pd.DataFrame({
    "тип до": housing.dtypes.astype(str),
    "тип после": optimized.dtypes.astype(str),
    "байт до": housing.memory_usage(deep=True)[1:],
    "байт после": optimized.memory_usage(deep=True)[1:],
})
comparison["экономия, %"] = (
    (1 - comparison["байт после"] / comparison["байт до"]) * 100
).round(1)
comparison

,тип до,тип после,байт до,байт после,"экономия, %"
price,int64,int32,4360,2180,50.0
area,int64,int16,4360,1090,75.0
bedrooms,int64,int8,4360,545,87.5
bathrooms,int64,int8,4360,545,87.5
stories,int64,int8,4360,545,87.5
mainroad,str,bool,28263,545,98.1
guestroom,str,bool,27892,545,98.0
basement,str,bool,27986,545,98.1
hotwaterheating,str,bool,27820,545,98.0
airconditioning,str,bool,27967,545,98.1


In [20]:
print(f"до оптимизации:    {before / 1024:8.1f} КБ")
print(f"после оптимизации: {after / 1024:8.1f} КБ")
print(f"экономия:          {(before - after) / 1024:8.1f} КБ "
      f"({(1 - after / before) * 100:.1f}%)")
print(f"датафрейм стал легче в {before / after:.1f} раза")

до оптимизации:       221.9 КБ
после оптимизации:      9.4 КБ
экономия:             212.6 КБ (95.8%)
датафрейм стал легче в 23.7 раза


**Вывод.** Основную экономию дали не числа, а строки: шесть столбцов `yes`/`no`
занимали больше всего памяти, потому что pandas хранит каждую строку как отдельный
объект Python. Перевод их в `bool` сжимает значение до одного байта.
Числовые столбцы дали меньший, но бесплатный выигрыш — `int64` избыточен для
величин вроде числа спален. При этом ни одно значение не изменилось: проверки
`assert` в предыдущей ячейке сравнивают данные до и после.